In [ ]:
run_date = None
config_path = 'config/settings.yaml'


In [ ]:
%matplotlib inline
from pathlib import Path
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display

from qis_risk_report.data.loaders import load_returns, load_portfolio, load_weights, load_factors
from qis_risk_report.risk import metrics as m
from qis_risk_report.risk.attribution import attribute_all, contribution_decomposition, cumulative_contribution
from qis_risk_report.risk.scenarios import default_scenarios, replay_all, run_scenario, ShockParams
from qis_risk_report.risk.portfolio import (
    marginal_var, component_var, qis_portfolio_correlation, diversification_benefit
)

cfg = yaml.safe_load(Path(config_path).read_text(encoding='utf-8-sig'))
returns_df = load_returns(cfg['data']['returns_path'])
portfolio_returns_df = load_portfolio(cfg['data']['portfolio_returns_path'])
weights_df = load_weights(cfg['data']['weights_path'])
factors_df = load_factors(cfg['data']['factors_path'])
weights = weights_df.iloc[-1]

sub_cols = [c for c in returns_df.columns if c != 'total']
port_cols = list(portfolio_returns_df.columns)
port_weights = weights.reindex(port_cols).fillna(0)

_run_date = run_date or returns_df.index[-1].strftime('%Y-%m-%d')
print(f'Report date: {_run_date}  |  QIS: {returns_df.index[0].date()} to {returns_df.index[-1].date()}  |  {len(returns_df)} trading days')


# QIS Risk Report

## 1. Performance


In [ ]:
cum_rets = returns_df.apply(m.cumulative_return).mul(100)
dd_total = m.drawdown_series(returns_df['total']).mul(100)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

cum_rets.plot(ax=axes[0])
axes[0].set_title('Cumulative Returns (%)')
axes[0].set_ylabel('%')
axes[0].legend(fontsize=8)

axes[1].plot(returns_df.index, dd_total, color='firebrick', linewidth=1)
axes[1].fill_between(returns_df.index, dd_total, 0, alpha=0.25, color='firebrick')
axes[1].set_title('QIS Total Drawdown (%)')
axes[1].set_ylabel('%')

plt.tight_layout()
plt.show()


In [ ]:
rows = []
for col in returns_df.columns:
    s = returns_df[col]
    rows.append({
        'Component': col,
        'Ann. Return': f'{m.annualised_return(s):.2%}',
        'Volatility': f'{m.annualised_volatility(s):.2%}',
        'Sharpe': f'{m.sharpe_ratio(s):.2f}',
        'Sortino': f'{m.sortino_ratio(s):.2f}',
        'Max DD': f'{m.max_drawdown(s):.2%}',
        'DD Duration (d)': m.drawdown_duration(s),
    })
pd.DataFrame(rows).set_index('Component')


## 2. Risk Metrics


In [ ]:
rows = []
for col in returns_df.columns:
    s = returns_df[col]
    rows.append({
        'Component': col,
        'Hist. VaR 99%': f'{m.historical_var(s, 0.99):.3%}',
        'Param. VaR 99%': f'{m.parametric_var(s, 0.99):.3%}',
        'CVaR/ES 99%': f'{m.expected_shortfall(s, 0.99):.3%}',
        'Hist. VaR 95%': f'{m.historical_var(s, 0.95):.3%}',
        'CVaR/ES 95%': f'{m.expected_shortfall(s, 0.95):.3%}',
    })
pd.DataFrame(rows).set_index('Component')


In [ ]:
corr = m.correlation_matrix(returns_df)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title('QIS Subcomponent Correlation Matrix')
plt.tight_layout()
plt.show()


## 3. Attribution


In [ ]:
attribution = attribute_all(returns_df, factors_df, window=60)
total_col = 'total' if 'total' in attribution else list(attribution.keys())[0]
total_attr = attribution[total_col]
betas = total_attr.drop(columns='r_squared').dropna()

if not betas.empty:
    fig, ax = plt.subplots(figsize=(12, 4))
    betas.plot(ax=ax)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title('Rolling 60-day Factor Betas — QIS Total')
    ax.set_ylabel('Beta')
    plt.tight_layout()
    plt.show()
    print('Latest factor betas:')
    display(total_attr.iloc[-1].rename('value').to_frame())
else:
    print('Insufficient history for rolling attribution (need ≥ 60 days).')


In [ ]:
cum_contrib = cumulative_contribution(returns_df).mul(100)
fig, ax = plt.subplots(figsize=(12, 4))
cum_contrib[sub_cols].plot(ax=ax)
ax.set_title('Cumulative Return Contribution by Component (%)')
ax.set_ylabel('%')
plt.tight_layout()
plt.show()


## 4. Scenarios


In [ ]:
results = replay_all(default_scenarios(), returns_df)
rows = []
for r in results:
    row = {'Scenario': r.name, 'Total P&L': f'{r.pnl_total:.2%}'}
    for k, v in r.pnl_by_component.items():
        row[k] = f'{v:.2%}'
    row['Days'] = r.metadata.get('n_days', 0)
    rows.append(row)
pd.DataFrame(rows).set_index('Scenario')


In [ ]:
shock_specs = [
    ('Parallel -1% Spot Shock', ShockParams(spot_shock=-0.01)),
    ('+50% Volatility Scaling', ShockParams(vol_shock=0.5)),
    ('80% Correlation Shock', ShockParams(corr_shock=0.8)),
]
rows = []
for label, shock in shock_specs:
    r = run_scenario(returns_df[sub_cols], shock)
    row = {'Scenario': label, 'Total P&L': f'{r.pnl_total:.2%}'}
    for k, v in r.pnl_by_component.items():
        row[k] = f'{v:.2%}'
    rows.append(row)
pd.DataFrame(rows).set_index('Scenario')


## 5. Portfolio Risk Contribution


In [ ]:
mv = marginal_var(portfolio_returns_df, port_weights, confidence=0.95)
cv = component_var(portfolio_returns_df, port_weights, confidence=0.95)

risk_contrib = pd.DataFrame({
    'Marginal VaR': mv.map(lambda x: f'{x:.4%}'),
    'Component VaR': cv.map(lambda x: f'{x:.4%}'),
    '% of Total VaR': (cv / cv.sum()).map(lambda x: f'{x:.1%}'),
})
display(risk_contrib)

try:
    div_benefit = diversification_benefit(portfolio_returns_df, port_weights, confidence=0.95)
    print(f'\nDiversification Benefit: {div_benefit:.4%}')
except ValueError as exc:
    print(f'\nDiversification benefit: {exc}')


In [ ]:
qis_corr = qis_portfolio_correlation(returns_df, portfolio_returns_df, weights=port_weights)
fig, ax = plt.subplots(figsize=(8, 3))
qis_corr.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('QIS Component Correlation with Portfolio Aggregate')
ax.set_ylabel('Pearson Correlation')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()
